# Casa 2. Ampliar la evaluación

**Duración orientativa: 45 min**

Objetivo: ampliar la evaluación con un conjunto de preguntas sintético, comparar nuestro judge con Ragas y
medir dos sesgos conocidos del judge: posición y longitud.

Referencias:
- Ragas: https://arxiv.org/abs/2309.15217
- Sesgos de LLM-as-a-judge (posición, longitud, autopreferencia): https://arxiv.org/abs/2306.05685

In [1]:
from rag_lab import CFG, set_seeds, summary

# CFG.FAST_MODE = False  # corpus de control completo (por defecto se lee FAST_MODE de .env)
# CFG.DOC_PATH = CFG.DATA_DIR / "mi_documento.pdf"  # cualquier PDF sirve
set_seeds()
summary()

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


## 1. Conjunto sintético

Pedimos al generador dos preguntas nuevas por página para 8 páginas al azar. Como el conjunto de clase,
es un borrador: solo sirve para ver si las conclusiones se mantienen con más preguntas.

In [2]:
import random

import pandas as pd
from rag_lab import evaluation, judge, llm, pdf, versions
from rag_lab.questions import QUESTION_SCHEMA, RULES

pages = pdf.clean_pages()
rng = random.Random(CFG.SEED)
selected = sorted(rng.sample(sorted(pages), 8))
synthetic = []
for n in selected:
    for attempt in range(2):
        fmt = {"type": "json_schema", "json_schema": {"name": "pregunta", "strict": True, "schema": QUESTION_SCHEMA}}
        prompt = f"{RULES}\nEscribe la pregunta número {attempt + 1} distinta de otras posibles.\n\nPÁGINA {n}:\n{pages[n][:6000]}"
        payload = llm.chat([{"role": "user", "content": prompt}], CFG.GEN_MODEL, response_format=fmt,
                         temperature=0.7, max_tokens=400, step="sinteticas").json()
        synthetic.append({"id": f"s{len(synthetic) + 1:02d}", "pregunta": payload["pregunta"], "paginas": [n],
                           "tipo": "tabla" if "|" in pages[n] else "texto"})
pd.DataFrame(synthetic)[["id", "paginas", "tipo", "pregunta"]]

,id,paginas,tipo,pregunta
0,s01,[1],texto,¿Cuál es el centro coordinador del proyecto de...
1,s02,[1],texto,¿Cuál es el centro coordinador del proyecto de...
2,s03,[2],texto,¿Qué línea se centra en la promoción del equil...
3,s04,[2],texto,¿Qué ámbito de competencias profesionales fome...
4,s05,[3],texto,¿Cuál es la duración establecida para el proye...
5,s06,[3],texto,Cuál es la duración total del proyecto según e...
6,s07,[4],texto,¿Cuál es el objetivo principal de la primera j...
7,s08,[4],texto,¿Qué objetivo tiene la primera justificación d...
8,s09,[8],texto,¿Qué nombre recibe la Tarea 6 dentro de la Lín...
9,s10,[8],texto,¿Qué tarea consiste en organizar sesiones regu...


In [3]:
rows = {}
for version in ["V1", "V3", "V6"]:
    rows[version] = evaluation.evaluate_rag(versions.build(version), synthetic, with_judge=False, register=False)["resumen"]
pd.DataFrame(rows).T[["recall@5", "recall@10", "MRR@10", "latencia_p50_ms"]].round(3)

,recall@5,recall@10,MRR@10,latencia_p50_ms
V1,0.9375,1.0,0.692187,872.455
V3,1.0,1.0,0.770833,756.84
V6,1.0,1.0,1.0,3186.045


## 2. Nuestro judge frente a Ragas

En un notebook ya hay un bucle asíncrono en marcha, así que usamos `await ...ascore()`.
Ragas calcula la fidelidad como la fracción de afirmaciones de la respuesta que el contexto respalda (0 a 1) y la
relevancia del contexto con su propio prompt. Usamos el mismo modelo judge por OpenRouter. Ragas guarda sus
llamadas en `cache/ragas`; su coste no aparece en nuestro registro.

In [4]:
import os

from openai import AsyncOpenAI
from ragas.cache import DiskCacheBackend
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRelevance, Faithfulness

client = AsyncOpenAI(base_url=CFG.OPENROUTER_BASE_URL, api_key=os.environ["OPENROUTER_API_KEY"])
# Los modelos que razonan (como gpt-5-mini) necesitan más max_tokens que el valor por defecto de Ragas (1024)
llm_ragas = llm_factory(CFG.JUDGE_MODEL, client=client, max_tokens=8000,
                        cache=DiskCacheBackend(cache_dir=str(CFG.CACHE_DIR / "ragas")))
faithfulness_ragas, context_ragas = Faithfulness(llm=llm_ragas), ContextRelevance(llm=llm_ragas)

questions = evaluation.load_questions()
detail = evaluation.evaluate_rag(versions.build("V3"), questions, register=False)["detalle"]
comparison = []
for _, row in detail.iterrows():
    contexts = row["contexto"].split("\n\n[Fragmento ")
    comparison.append({
        "id": row["id"], "tipo": row["tipo"],
        "fidelidad (nuestro)": row["fidelidad"],
        "fidelidad (Ragas)": (await faithfulness_ragas.ascore(user_input=row["pregunta"], response=row["respuesta"],
                                                           retrieved_contexts=contexts)).value,
        "contexto (nuestro)": row["relevancia_contexto"],
        "contexto (Ragas)": (await context_ragas.ascore(user_input=row["pregunta"], retrieved_contexts=contexts)).value,
    })
comparison = pd.DataFrame(comparison).set_index("id")
comparison

,tipo,fidelidad (nuestro),fidelidad (Ragas),contexto (nuestro),contexto (Ragas)
id,,,,,
q01,texto,1,1.0,1,1.0
q02,texto,1,1.0,1,1.0
q03,texto,1,1.0,1,1.0
q04,texto,1,1.0,1,1.0
q05,texto,1,1.0,1,1.0
q06,texto,1,1.0,1,1.0
q07,texto,1,1.0,1,1.0
q08,tabla,1,1.0,1,1.0
q09,tabla,1,1.0,1,1.0


In [5]:
# Acuerdo binarizando Ragas en 0,5
for ours_col, ragas_col in [("fidelidad (nuestro)", "fidelidad (Ragas)"), ("contexto (nuestro)", "contexto (Ragas)")]:
    print(ours_col, judge.agreement(comparison[ours_col].astype(int).tolist(),
                                 (comparison[ragas_col] >= 0.5).astype(int).tolist()))

fidelidad (nuestro) {'n': 20, 'acuerdo_%': 95.0, 'kappa': 0.8275862068965517}
contexto (nuestro) {'n': 20, 'acuerdo_%': 100.0, 'kappa': 1.0}


## 3. Sesgo de posición

Damos al judge dos respuestas (de V0 y de V7) a la misma pregunta y le pedimos cuál es mejor; después
intercambiamos el orden. Un judge sin sesgo de posición elige la misma respuesta en los dos órdenes.

In [6]:
PAIR_SCHEMA = {"type": "object", "properties": {"razonamiento": {"type": "string"},
                                                "mejor": {"type": "string", "enum": ["A", "B", "empate"]}},
               "required": ["razonamiento", "mejor"], "additionalProperties": False}

def compare(question, a, b):
    prompt = (f"¿Qué respuesta contesta mejor a la pregunta? Responde A, B o empate.\n\nPREGUNTA: {question}\n\n"
              f"RESPUESTA A:\n{a}\n\nRESPUESTA B:\n{b}")
    return llm.judge(prompt, PAIR_SCHEMA, step="judge_pares").json()["mejor"]

d0 = evaluation.evaluate_rag(versions.build("V0"), questions, with_judge=False, register=False)["detalle"]
from rag_lab import multimodal
structured = versions.pdf_fragments("cabeceras")
enriched = versions.enrich(structured, versions.document_title(pages))
d7 = evaluation.evaluate_rag(multimodal.Multimodal(enriched, multimodal.describe_pages(), "siglip"),
                            questions, with_judge=False, register=False)["detalle"]

position = []
for (_, x), (_, y) in zip(d0.iterrows(), d7.iterrows()):
    if x["respuesta"] == y["respuesta"]:
        continue
    order_1 = compare(x["pregunta"], x["respuesta"], y["respuesta"])  # A = V0
    order_2 = compare(x["pregunta"], y["respuesta"], x["respuesta"])  # A = V7
    winner_1 = {"A": "V0", "B": "V7"}.get(order_1, "empate")
    winner_2 = {"A": "V7", "B": "V0"}.get(order_2, "empate")
    position.append({"id": x["id"], "orden V0-V7": winner_1, "orden V7-V0": winner_2, "consistente": winner_1 == winner_2})
position = pd.DataFrame(position)
print(f"Pares distintos: {len(position)} | consistentes al cambiar el orden: {position['consistente'].mean():.0%}")
position

Pares distintos: 14 | consistentes al cambiar el orden: 57%


,id,orden V0-V7,orden V7-V0,consistente
0,q01,empate,V7,False
1,q03,V7,V7,True
2,q04,empate,empate,True
3,q05,empate,empate,True
4,q06,V7,V0,False
5,q07,V0,V0,True
6,q08,V7,V0,False
7,q09,empate,empate,True
8,q10,empate,V7,False
9,q11,V7,V7,True


## 4. Sesgo de longitud

Alargamos cada respuesta de V3 con una frase de relleno que no aporta información y preguntamos al judge cuál
prefiere (en los dos órdenes, para no mezclar con el sesgo de posición). Sin sesgo, debería elegir empate o la corta.

In [7]:
FILLER = (" Esta respuesta se ha elaborado a partir de la documentación disponible del proyecto, teniendo en cuenta "
           "el contexto general de la memoria y los distintos apartados que la componen.")
lengths = []
for _, row in detail.iterrows():
    short_answer, long_answer = row["respuesta"], row["respuesta"] + FILLER
    r1 = {"A": "corta", "B": "larga"}.get(compare(row["pregunta"], short_answer, long_answer), "empate")
    r2 = {"A": "larga", "B": "corta"}.get(compare(row["pregunta"], long_answer, short_answer), "empate")
    lengths.append({"id": row["id"], "orden corta-larga": r1, "orden larga-corta": r2})
lengths = pd.DataFrame(lengths)
print(pd.concat([lengths["orden corta-larga"], lengths["orden larga-corta"]]).value_counts())

empate    15
larga     15
corta     10
Name: count, dtype: int64


Prueba: cambia `CFG.JUDGE_MODEL` por otro modelo (de una familia distinta al generador) y repite las secciones 3 y 4.
¿Cambian los sesgos?